# 柳小乐 v51 — SFT Stage 1：纯 SFT 对话训练
## 从 v51 Base checkpoint (step_015000) 续训

**架构：** StdAttn + GQA 4KV + QK-Norm + SwiGLU · 8K MiniMind BPE · Cosine LR
**数据：** data_stage1/sft_pt/*.pt · ~1.74M 条（已抽取 163,620 条给 Stage 2）· 1 epoch ≈ 13,592 步
**Base：** checkpoints_v51/step_015000.pt（MiniMind 100% 预训练 15,000 步）

**策略：**
- 纯 SFT 对话训练，不含身份数据（身份留给 Stage 2）
- 统一 assistant：标签，masked loss（只计算 assistant 回复）
- 1→0 过渡处理：assistant 回复后 2 个 token 也参与训练
- ★ 参考 pretrain_v5.1.ipynb 训练模式：Cosine LR + warmup + 断点续跑

**前置（仅一次）：**
1. 运行 `preprocess_pretrain_data.ipynb`（清洗 + tokenize → data_stage1/sft_pt/*.pt）
2. Run All 本 notebook</cell id="cell-0">


In [ ]:
# Cell 1：环境 + 路径检测

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:512'

import torch
import torch.nn as nn
import torch.nn.functional as F
import gc
import time
import math
import json
from pathlib import Path

# 自动适配本地/DSW 路径
BASE = Path.cwd()
if not (BASE / "tokenizer_minimind_8k").exists() and (BASE / "shayler2.0").exists():
    BASE = BASE / "shayler2.0"

print(f"PyTorch: {torch.__version__}")
print(f"ROCm: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) or 'MI300X'}")
mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"显存总量: {mem:.1f} GB")
print(f"项目根目录: {BASE}")

def print_memory_stats(prefix=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        free_mem = torch.cuda.mem_get_info()[0] / 1024**3
        print(f"{prefix}显存: {allocated:.2f}GB 已用 / {reserved:.2f}GB 预留 / {free_mem:.2f}GB 空闲")

print_memory_stats("初始")
print("Cell 1 OK")

In [ ]:
# Cell 2：路径 + CONFIG + TC（★ 参考 pretrain_v5.1 · 纯 SFT Stage 1）

# ── 路径（与 pretrain_v5.1 对齐）──
TOKENIZER = BASE / "tokenizer_minimind_8k" / "tokenizer.json"
BASE_CKPT_DIR = BASE / "checkpoints_v51"                 # ★ v51 Base（加载 step_015000.pt）
SFT_CKPT_DIR = BASE / "checkpoints_sft1_v51"             # ★ SFT Stage 1 checkpoint（保存）
SFT_CKPT_DIR.mkdir(parents=True, exist_ok=True)
PT_DIR = BASE / "data_stage1" / "sft_pt"                 # ★ preprocess_pretrain_data.ipynb 输出

# ── 分词器信息 ──
from tokenizers import Tokenizer as TokReader
tok = TokReader.from_file(str(TOKENIZER))
VOCAB_SIZE = tok.get_vocab_size()
print(f"分词器词表: {VOCAB_SIZE}")

# ═══════════════════════════════════════════════
# CONFIG：与 pretrain_v5.1 完全一致（StdAttn + GQA 4KV + QK-Norm + SwiGLU）
# ═══════════════════════════════════════════════
CONFIG = {
    "vocab_size": VOCAB_SIZE,
    "n_layer": 20,
    "n_head": 16,
    "n_query_groups": 4,
    "n_embd": 1024,
    "intermediate_size": 3584,
    "block_size": 1024,
    "norm_eps": 1e-5,
    "diff_attention": False,
    "qk_norm": True,
}

# ═══════════════════════════════════════════════
# TC：SFT Stage 1 训练超参（与 pretrain_v5.1 模式对齐）
# ═══════════════════════════════════════════════
TC = {
    "max_steps": 13592,             # ★ ~1 epoch（1,739,692 ÷ 128 = 13,591.3）
    "warmup": 200,                  # ~1.5% warmup
    "min_lr_ratio": 0.1,            # Cosine 衰减至峰值 10%

    "micro_batch_size": 64,         # 有效 batch = 64×2 = 128
    "grad_accum": 2,

    "max_seq_len": 1024,

    "lr": 5e-5,                     # ★ SFT 学习率（Base 1/6：3e-4 → 5e-5）
    "weight_decay": 0.01,
    "max_time": int(7.9 * 3600),    # 单 session 8h 限制
    "save_interval": 500,           # 每 500 步保存 → ~27 个 ckpt

    "empty_cache_steps": 200,       # 与 pretrain_v5.1 一致
    "gc_collect": True,
}

device = torch.cuda.current_device()
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision('medium')  # ★ AMD MI300X：medium 最优（与 pretrain_v5.1 一致）

# ★ 确保 SDPA 优先使用 Flash Attention 后端
if hasattr(torch.backends.cuda, 'enable_flash_sdp'):
    torch.backends.cuda.enable_flash_sdp(True)
    print("★ Flash SDP 已启用")

effective_batch = TC["micro_batch_size"] * TC["grad_accum"]

print(f"模型: {CONFIG['n_layer']}层 x {CONFIG['n_embd']}维 ≈ 294M")
print(f"Batch: {TC['micro_batch_size']}×{TC['grad_accum']}={effective_batch}")
print(f"步数: {TC['max_steps']} (~1 epoch) | lr={TC['lr']:.1e} | warmup={TC['warmup']}")
print(f"★ Cosine LR → min_lr={TC['lr']*TC['min_lr_ratio']:.1e}")
print(f"★ 纯 SFT Stage 1 · masked loss (assistant only) · 与 pretrain_v5.1 架构一致")
print(f"★ Base: {BASE_CKPT_DIR} | 数据: {PT_DIR}")
print("Cell 2 OK")

In [ ]:
# Cell 3：数据确认（.pt 文件由 preprocess_pretrain_data.ipynb 生成）

if not PT_DIR.exists() or not list(PT_DIR.glob("sft_*.pt")):
    raise FileNotFoundError(
        f"❌ .pt 文件不存在: {PT_DIR}\n"
        f"请先 Run All preprocess_pretrain_data.ipynb"
    )

pt_files = sorted(PT_DIR.glob("sft_*.pt"))
total_mb = sum(f.stat().st_size for f in pt_files) / 1e6

# 估算总条数
n_first = torch.load(pt_files[0], map_location="cpu")["tokens"].shape[0]
est_total = n_first * len(pt_files)
print(f"✅ .pt 文件就绪: {PT_DIR}")
print(f"   文件数: {len(pt_files)} · {total_mb:.0f} MB")
print(f"   估算总条数: ~{est_total:,} (~{est_total * 1024 / 1e9:.1f}B tokens)")

# 解码 1 条验证
from tokenizers import Tokenizer as _Tok
_tok = _Tok.from_file(str(TOKENIZER))
data = torch.load(pt_files[0], map_location="cpu")
sample_tokens = data["tokens"][0].tolist()
sample_mask = data["masks"][0].tolist()
valid_tokens = [t for t in sample_tokens if t > 0]
decoded = _tok.decode(valid_tokens)
assist_n = int(sum(sample_mask))
print(f"   样本: {len(valid_tokens)} tokens | assist={assist_n}/{len(sample_mask)}")
print(f"   解码: {decoded[:150]}...")
print("Cell 3 OK")

In [ ]:
# Cell 4：模型架构（与 v31 Base 一致 — StdAttn + GQA + QK-Norm + SwiGLU）
# ★ 优化：SDPA Flash Attention 替代手动 softmax

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x / rms) * self.weight


def apply_rope(x, cos, sin):
    # cos/sin: [1, 1, T, hd//2]，x: [B, nh, T, hd]
    # 直接 broadcast，不需要额外 unsqueeze（v31 fix）
    r = x.float().reshape(*x.shape[:-1], -1, 2)
    out0 = r[..., 0] * cos - r[..., 1] * sin
    out1 = r[..., 1] * cos + r[..., 0] * sin
    return torch.stack([out0, out1], dim=-1).flatten(-2).to(x.dtype)


class StdAttn(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.nh = c["n_head"]
        self.hd = c["n_embd"] // c["n_head"]
        self.ng = c.get("n_query_groups", self.nh)
        self.scale = self.hd ** -0.5
        kv_dim = self.hd * self.ng
        self.q_proj = nn.Linear(c["n_embd"], c["n_embd"], bias=False)
        self.k_proj = nn.Linear(c["n_embd"], kv_dim, bias=False)
        self.v_proj = nn.Linear(c["n_embd"], kv_dim, bias=False)
        self.o_proj = nn.Linear(c["n_embd"], c["n_embd"], bias=False)
        if c.get("qk_norm", False):
            self.q_norm = RMSNorm(self.hd)
            self.k_norm = RMSNorm(self.hd)
        else:
            self.q_norm = nn.Identity()
            self.k_norm = nn.Identity()

    def forward(self, x, cos, sin, mask=None):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.nh, self.hd)
        k = self.k_proj(x).view(B, T, self.ng, self.hd)
        v = self.v_proj(x).view(B, T, self.ng, self.hd)
        rp = self.nh // self.ng
        if rp > 1:
            k = k.repeat_interleave(rp, dim=2)
            v = v.repeat_interleave(rp, dim=2)
        q = self.q_norm(q)
        k = self.k_norm(k)
        q = q.transpose(1, 2); k = k.transpose(1, 2); v = v.transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        # ★ SDPA Flash Attention（比手动 softmax 快 + 省显存）
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(out)


class SwiGLU(nn.Module):
    def __init__(self, c):
        super().__init__()
        d = c["n_embd"]; i = c["intermediate_size"]
        self.w1 = nn.Linear(d, i, bias=False)
        self.w2 = nn.Linear(d, i, bias=False)
        self.w3 = nn.Linear(i, d, bias=False)

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))


class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.attn_norm = RMSNorm(c["n_embd"], c["norm_eps"])
        self.attn = StdAttn(c)
        self.ffn_norm = RMSNorm(c["n_embd"], c["norm_eps"])
        self.ffn = SwiGLU(c)

    def forward(self, x, cos, sin, mask=None):
        x = x + self.attn(self.attn_norm(x), cos, sin)
        x = x + self.ffn(self.ffn_norm(x))
        return x


class Shayler(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.cfg = c; self.bs = c["block_size"]
        self.emb = nn.Embedding(c["vocab_size"], c["n_embd"])
        self.layers = nn.ModuleList([Block(c) for _ in range(c["n_layer"])])
        self.norm = RMSNorm(c["n_embd"], c["norm_eps"])
        self.head = nn.Linear(c["n_embd"], c["vocab_size"], bias=True)
        self.emb.weight = self.head.weight

        freqs = 1.0 / (10000 ** (torch.arange(0, c["n_embd"] // c["n_head"], 2).float() / (c["n_embd"] // c["n_head"])))
        t = torch.arange(self.bs).float()
        a = torch.outer(t, freqs)
        self.register_buffer("rc", torch.cos(a).unsqueeze(0).unsqueeze(0))
        self.register_buffer("rs", torch.sin(a).unsqueeze(0).unsqueeze(0))

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.emb(idx)
        cos = self.rc[:, :, :T].to(x.device)
        sin = self.rs[:, :, :T].to(x.device)

        for layer in self.layers:
            x = layer(x, cos, sin)  # SDPA is_causal 内部处理 mask

        x = self.norm(x)
        logits = self.head(x)

        if targets is not None:
            return F.cross_entropy(logits.view(-1, self.cfg["vocab_size"]), targets.view(-1))
        return logits

print("创建模型...")
model = Shayler(CONFIG)
print(f"参数: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")
print("架构: StdAttn + GQA(4KV) + QK-Norm + SwiGLU + Flash Attention (SDPA)")
print("Cell 4 OK")

In [ ]:
# Cell 5：分词器 + Chat Template（★ 统一 assistant：标签 · 身份 QA 与 SFT 格式一致）

from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(str(TOKENIZER))
print(f"词表大小: {tokenizer.get_vocab_size()}")

# ★ 统一标签：SFT .pt 和身份 QA 都用 assistant：
ROLE_TAGS = {
    "user": "user：",
    "assistant": "assistant：",
}

def format_chat(messages, max_len=1024):
    """
    将 messages 列表编码为 (token_ids, loss_mask)
    ★ 统一 assistant：标签，与 SFT .pt 格式一致
    ★ 只训练 assistant 回复部分（mask=1.0）
    """
    tokens, loss_mask = [], []

    for msg in messages:
        role = msg.get("role", "")
        content = msg.get("content", "")

        if role not in ROLE_TAGS:
            continue
        if not content:
            continue

        tag = ROLE_TAGS[role]
        tag_ids = tokenizer.encode(tag).ids
        content_ids = tokenizer.encode(content).ids
        ids = tag_ids + content_ids

        tokens.extend(ids)
        if role == "assistant":
            loss_mask.extend([1.0] * len(ids))
        else:
            loss_mask.extend([0.0] * len(ids))

    # 1→0 过渡处理
    for i in range(1, len(loss_mask)):
        if loss_mask[i] == 0.0 and loss_mask[i-1] == 1.0:
            loss_mask[i] = 1.0
            if i + 1 < len(loss_mask):
                loss_mask[i+1] = 1.0

    if len(tokens) > max_len:
        tokens = tokens[:max_len]
        loss_mask = loss_mask[:max_len]

    return tokens, loss_mask

# 快速测试
test_msgs = [
    {"role": "user", "content": "你好"},
    {"role": "assistant", "content": "嗨~今天天气不错！"},
]
t, m = format_chat(test_msgs)
decoded = tokenizer.decode(t)
assist_n = int(sum(m))
print(f"编码测试: {decoded[:100]}... | assist_tokens={assist_n}/{len(m)}")
print(f"★ 统一标签: user：/ assistant： — SFT .pt 和身份 QA 格式完全一致")
print("Cell 5 OK")

In [ ]:
# Cell 6：SFTDataset — 加载 .pt 文件（★ 参考 pretrain_v5.1 Dataset 模式 · 简化版）

class SFTDataset(torch.utils.data.IterableDataset):
    """.pt 文件随机打乱读取，支持断点续读（skip_samples）"""
    def __init__(self, pt_dir, skip_samples=0):
        self.pt_dir = Path(pt_dir)
        self.files = sorted(self.pt_dir.glob("sft_*.pt"))
        if not self.files:
            raise FileNotFoundError(f"无 .pt 文件: {pt_dir}")

        # 统计总量
        n_first = torch.load(self.files[0], map_location="cpu")["tokens"].shape[0]
        self.total_samples = n_first * len(self.files)  # 近似（最后文件可能少）
        self.skip_samples = skip_samples

        total_mb = sum(f.stat().st_size for f in self.files) / 1e6
        print(f"SFT 数据: {len(self.files)} 个 .pt · {total_mb:.0f} MB · ~{self.total_samples:,} 条")
        if skip_samples > 0:
            print(f"  ⏭ 续读: 跳过 {skip_samples:,} 条")

    def __iter__(self):
        import random as _random
        need_skip = self.skip_samples

        while True:
            files = self.files.copy()
            _random.shuffle(files)

            for fp in files:
                data = torch.load(fp, map_location="cpu")
                tokens = data["tokens"]   # (N, 1024) int16
                masks = data["masks"]     # (N, 1024) bool

                idx = torch.randperm(tokens.shape[0])

                # 断点续读
                if need_skip > 0:
                    if need_skip >= len(idx):
                        need_skip -= len(idx)
                        continue
                    else:
                        idx = idx[need_skip:]
                        need_skip = 0

                for i in idx:
                    yield tokens[i].long(), masks[i].float()

            # 一轮结束后重置 skip
            self.skip_samples = 0
            need_skip = 0


print("✅ SFTDataset 就绪（简化版 · 对齐 pretrain_v5.1 模式）")
print("Cell 6 OK")

In [ ]:
# Cell 7：加载 v51 Base Checkpoint + SFT Stage 1 训练循环（★ 参考 pretrain_v5.1 模式）

TOTAL_STEPS = TC["max_steps"]           # 13592

# ———— Loss 日志 ————
LOSS_LOG = SFT_CKPT_DIR / "loss_log.csv"
_loss_log_first_write = not LOSS_LOG.exists() or LOSS_LOG.stat().st_size == 0

print("迁移模型到 GPU...")
model.cuda().train()
print_memory_stats("模型迁移后")

# ———— 数据检查 ————
print(f"\n数据检查:")
if PT_DIR.exists():
    n_pt = len(list(PT_DIR.glob("sft_*.pt")))
    total_mb = sum(f.stat().st_size for f in PT_DIR.glob("sft_*.pt")) / 1e6
    print(f"  {PT_DIR.name}: {n_pt} 个 .pt · {total_mb:.0f} MB ✓")
else:
    print(f"  ⚠️ {PT_DIR} 不存在！请先运行 preprocess_pretrain_data.ipynb")

# ———— 续跑检测 ————
sft_ckpts = sorted(SFT_CKPT_DIR.glob("sft_step_*.pt"))
start_step = 0
lema = None
consumed_samples = 0

if sft_ckpts:
    # ★ 从 SFT checkpoint 续跑
    resume_ckpt = sft_ckpts[-1]
    print(f"\n{'='*50}")
    print(f"📂 SFT 续跑: {resume_ckpt.name}")
    sd = torch.load(resume_ckpt, map_location="cpu")
    model.load_state_dict(sd["model"])
    start_step = sd.get("step", 0)
    lema = sd.get("loss_ema", None)
    consumed_samples = sd.get("consumed_samples", start_step * effective_batch)
    print(f"   step={start_step}  loss_ema={lema:.4f}" if lema else f"   step={start_step}")
    print(f"   consumed_samples={consumed_samples:,}")

    opt = torch.optim.AdamW(model.parameters(), lr=TC['lr'],
                            betas=(0.9, 0.95), weight_decay=TC['weight_decay'])
    if "optimizer" in sd:
        try:
            opt.load_state_dict(sd["optimizer"])
            print("   ✅ 优化器状态已恢复")
        except Exception:
            print("   ⚠️ 优化器状态不兼容，重新初始化")
        for pg in opt.param_groups:
            pg["lr"] = TC["lr"]
            pg["weight_decay"] = TC["weight_decay"]

    remaining = TOTAL_STEPS - start_step
    print(f"   剩余: {remaining} 步")
    del sd
    torch.cuda.empty_cache()
    print(f"{'='*50}")
else:
    # ★ 从 v51 Base checkpoint 开始 SFT Stage 1
    base_ckpts = sorted(BASE_CKPT_DIR.glob("step_*.pt"))
    if not base_ckpts:
        raise FileNotFoundError(f"{BASE_CKPT_DIR} 下无 Base checkpoint！")

    # 优先使用 step_015000.pt，否则用最新
    target_ckpt = BASE_CKPT_DIR / "step_015000.pt"
    if target_ckpt.exists():
        base_ckpt = target_ckpt
    else:
        base_ckpt = base_ckpts[-1]

    print(f"\n{'='*50}")
    print(f"🆕 从 v51 Base 开始 SFT Stage 1")
    print(f"   Base checkpoint: {base_ckpt.name}")

    sd = torch.load(base_ckpt, map_location="cpu")

    # ★ 形状校验（确保架构完全匹配 pretrain_v5.1）
    print(f"\n  形状校验:")
    checks = [
        ("emb.weight", (VOCAB_SIZE, CONFIG["n_embd"])),
        ("head.weight", (VOCAB_SIZE, CONFIG["n_embd"])),
        ("layers.0.attn.q_proj.weight", (CONFIG["n_embd"], CONFIG["n_embd"])),
        ("layers.19.ffn.w3.weight", (CONFIG["n_embd"], CONFIG["intermediate_size"])),
    ]
    all_ok = True
    for name, expected in checks:
        if name in sd["model"]:
            actual = sd["model"][name].shape
            ok = actual == expected
            if not ok:
                all_ok = False
            print(f"   {'✅' if ok else '❌'} {name}: {list(actual)} (期望 {list(expected)})")
        else:
            print(f"   ⚠️ {name}: 不在 checkpoint 中")
            all_ok = False

    if not all_ok:
        raise RuntimeError("❌ 架构不匹配！检查 CONFIG 是否与 pretrain_v5.1 一致")

    model.load_state_dict(sd["model"])
    base_step = sd.get("step", "?")
    base_ema = sd.get("loss_ema", "?")
    print(f"\n   Base step: {base_step}  ema: {base_ema}")
    del sd
    torch.cuda.empty_cache()

    opt = torch.optim.AdamW(model.parameters(), lr=TC['lr'],
                            betas=(0.9, 0.95), weight_decay=TC['weight_decay'])
    print(f"   Optimizer: 全新初始化 (LR={TC['lr']:.1e})")
    print(f"{'='*50}")

# ———— DataLoader（★ 简化版 SFTDataset · 对齐 pretrain_v5.1）————
print("\n初始化 DataLoader...")
ds = SFTDataset(PT_DIR, skip_samples=consumed_samples)
dl = iter(torch.utils.data.DataLoader(
    ds, batch_size=TC["micro_batch_size"],
    num_workers=0, pin_memory=False))

steps_per_epoch = ds.total_samples // effective_batch
print(f"数据配置: ~{ds.total_samples:,} 条 → 1 epoch ≈ {steps_per_epoch:,} steps")
print(f"         {TOTAL_STEPS:,} steps ≈ {TOTAL_STEPS/steps_per_epoch:.1f} epochs")
print(f"有效 Batch: {TC['micro_batch_size']}×{TC['grad_accum']}={effective_batch}")
print("DataLoader 就绪")

# ★ 训练前清理
torch.cuda.empty_cache()
gc.collect()
print_memory_stats("训练前")

# ═══════════════════════════════════════════════
# SFT Stage 1 训练循环（★ 参考 pretrain_v5.1 模式）
# ═══════════════════════════════════════════════
total = TOTAL_STEPS
warmup = TC["warmup"]
max_t = TC["max_time"]
min_lr_ratio = TC["min_lr_ratio"]

print(f"\n{'='*50}")
print(f"SFT Stage 1: step {start_step} → {total}")
print(f"Base: {BASE_CKPT_DIR} | 数据: {PT_DIR}")
print(f"策略: 纯 SFT · masked loss (assistant only) · 1→0 过渡 2 token")
print(f"优化器: AdamW lr={TC['lr']:.1e} wd={TC['weight_decay']} betas=(0.9, 0.95)")
print(f"LR schedule: warmup {warmup}步 → cosine → {TC['lr']*min_lr_ratio:.1e}")
print(f"有效 Batch: {TC['micro_batch_size']}×{TC['grad_accum']}={effective_batch}")
print(f"💾 save_interval={TC['save_interval']}步 | 保留最近5个 checkpoint")
print(f"📋 loss 日志: {LOSS_LOG}")
if consumed_samples > 0:
    print(f"⏭ 数据续读: 跳过前 {consumed_samples:,} 样本")
print(f"{'='*50}")

# 初始化 loss 日志（与 pretrain_v5.1 格式一致）
_loss_file = open(LOSS_LOG, 'a')
if _loss_log_first_write:
    _loss_file.write("step,loss,ema,lr,grad_norm\n")
    print(f"📋 新建 loss 日志: {LOSS_LOG}")
else:
    print(f"📋 续写 loss 日志: {LOSS_LOG}")

step = start_step
t0 = time.time()
t_data = t_fwd = t_bwd = t_opt = 0.0
data_exhausted = False

while step < total and not data_exhausted:
    elapsed = time.time() - t0
    if elapsed > max_t:
        print(f"\n⏰ {elapsed/3600:.1f}h 到达 session 时限，保存退出...")
        break

    # ═══════════════════════════════════════════
    # Cosine LR schedule（与 pretrain_v5.1 完全一致）
    # ═══════════════════════════════════════════
    if step < warmup:
        lr_scale = step / max(warmup, 1)
    else:
        progress = (step - warmup) / max(total - warmup, 1)
        lr_scale = min_lr_ratio + (1 - min_lr_ratio) * 0.5 * (1 + math.cos(math.pi * progress))
    current_lr = TC["lr"] * lr_scale

    for pg in opt.param_groups:
        pg["lr"] = current_lr

    accum_loss = 0
    opt.zero_grad(set_to_none=True)

    for acc_step in range(TC["grad_accum"]):
        torch.cuda.synchronize()
        _t0 = time.time()
        # ★ 数据耗尽保护
        try:
            tokens, mask = next(dl)
        except StopIteration:
            data_exhausted = True
            break
        tokens = tokens.to(device, non_blocking=True)
        mask = mask.to(device, non_blocking=True)
        torch.cuda.synchronize()
        t_data += time.time() - _t0

        torch.cuda.synchronize()
        _t0 = time.time()
        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            # ★ SFT：不传 targets，手动计算 masked loss
            logits = model(tokens[:, :-1].contiguous())
            ce_per_token = F.cross_entropy(
                logits.view(-1, CONFIG["vocab_size"]),
                tokens[:, 1:].contiguous().view(-1),
                reduction='none'
            )
            ce_2d = ce_per_token.view(tokens.shape[0], -1)
            masked_ce = (ce_2d * mask[:, 1:]).sum() / mask[:, 1:].sum().clamp(min=1)
            loss = masked_ce / TC["grad_accum"]
        torch.cuda.synchronize()
        t_fwd += time.time() - _t0

        torch.cuda.synchronize()
        _t0 = time.time()
        loss.backward()
        torch.cuda.synchronize()
        t_bwd += time.time() - _t0

        accum_loss += masked_ce.item()
        del logits, ce_per_token, ce_2d, masked_ce, loss, tokens, mask

    # ★ 数据耗尽：保存退出
    if data_exhausted:
        print(f"\n⚠️ 数据耗尽 @ step {step}！保存后退出...")
        ckpt_path = SFT_CKPT_DIR / f"sft_step_{step:06d}_exhausted.pt"
        torch.save({
            "step": step,
            "model": model.state_dict(),
            "optimizer": opt.state_dict(),
            "loss_ema": lema,
            "config": CONFIG,
            "consumed_samples": step * effective_batch,
        }, ckpt_path)
        print(f"💾 {ckpt_path.name}")
        break

    torch.cuda.synchronize()
    _t0 = time.time()
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    torch.cuda.synchronize()
    t_opt += time.time() - _t0

    # ★ 显存管理（与 pretrain_v5.1 一致）
    if step % TC["empty_cache_steps"] == 0 and step > 0:
        torch.cuda.empty_cache()
        if TC["gc_collect"]:
            gc.collect(generation=2)
        torch.cuda.reset_peak_memory_stats()

    step += 1
    avg_loss = accum_loss / TC["grad_accum"]
    if lema is None:
        lema = avg_loss
    else:
        lema = 0.95 * lema + 0.05 * avg_loss

    # ★ 每步写 loss 日志（与 pretrain_v5.1 一致）
    _loss_file.write(f"{step},{avg_loss:.6f},{lema:.6f},{current_lr:.8f},{grad_norm:.4f}\n")
    _loss_file.flush()

    if step % 10 == 0:
        eh = (time.time() - t0) / 3600
        n_steps = step - start_step
        sps = n_steps / (time.time() - t0) if time.time() > t0 else 0
        rh = (total - step) / sps / 3600 if sps > 0 else 0
        mem_alloc = torch.cuda.memory_allocated() / 1024**3
        lr_tag = "[WARMUP]" if step <= warmup else "[COSINE]"
        total_t = t_data + t_fwd + t_bwd + t_opt
        avg_t = total_t / n_steps if n_steps > 0 else 0
        print(f"  {step}/{total}({step/total*100:.1f}%) loss={avg_loss:.4f} ema={lema:.4f} "
              f"{lr_tag} lr={current_lr:.1e} gn={grad_norm:.1f} "
              f"⏱{avg_t:.1f}s/步 | {eh:.1f}h 剩~{rh:.0f}h | Mem: {mem_alloc:.2f}GB")
        if n_steps > 0:
            print(f"    ⏱ data={t_data/n_steps*1000:.0f}ms fwd={t_fwd/n_steps*1000:.0f}ms bwd={t_bwd/n_steps*1000:.0f}ms opt={t_opt/n_steps*1000:.0f}ms")

    # ———— 保存 checkpoint（与 pretrain_v5.1 格式一致）————
    if step % TC["save_interval"] == 0:
        ckpt_path = SFT_CKPT_DIR / f"sft_step_{step:06d}.pt"
        torch.save({
            "step": step,
            "model": model.state_dict(),
            "optimizer": opt.state_dict(),
            "loss_ema": lema,
            "config": CONFIG,
            "consumed_samples": step * effective_batch,
        }, ckpt_path)
        # ★ 保留最近 5 个 checkpoint
        all_ckpts = sorted(SFT_CKPT_DIR.glob("sft_step_*.pt"), key=lambda p: p.stat().st_mtime)
        for old_ckpt in all_ckpts[:-5]:
            old_ckpt.unlink()
            print(f"  🗑️ 删除: {old_ckpt.name}")
        print(f"  💾 {ckpt_path.name} | step={step} ema={lema:.4f} lr={current_lr:.1e} | {(time.time()-t0)/3600:.1f}h")

# ———— 最终保存 ————
if step > 0 and step % TC["save_interval"] != 0 and not data_exhausted:
    ckpt_path = SFT_CKPT_DIR / f"sft_step_{step:06d}.pt"
    torch.save({
        "step": step,
        "model": model.state_dict(),
        "optimizer": opt.state_dict(),
        "loss_ema": lema,
        "config": CONFIG,
        "consumed_samples": step * effective_batch,
    }, ckpt_path)
    print(f"\n💾 最终保存: {ckpt_path.name} | step={step} ema={lema:.4f}")

_loss_file.close()
print(f"📋 loss 日志已关闭: {LOSS_LOG}")

if data_exhausted:
    print(f"\n⚠️ 数据耗尽 @ step {step} | ema={lema:.4f}")
    print(f"   建议: 增加 epochs 或扩展数据集后续跑")
elif step >= total:
    print(f"\n🎉 SFT Stage 1 完成！")
    print(f"   终局: step={step} ema={lema:.4f} lr={current_lr:.1e}")
    print(f"   ★ 下一步: 用 chat.py 测试推理 → 进入 Stage 2 身份注入")
else:
    print(f"\n⏸ Session 结束 | step={step}/{total} ({step/total*100:.1f}%)")
    remaining = total - step
    sps = (step - start_step) / (time.time() - t0) if time.time() > t0 else 0
    if sps > 0:
        print(f"   剩余: {remaining} 步 → ~{remaining * sps / 3600:.1f}h")
    print(f"   下次续跑: 重新运行本 Cell 即可自动恢复")

In [ ]:
# Cell 8：进度查看（★ v51 SFT Stage 1）

try:
    _ = SFT_CKPT_DIR
except NameError:
    BASE = Path.cwd()
    if not (BASE / "tokenizer_minimind_8k").exists() and (BASE / "shayler2.0").exists():
        BASE = BASE / "shayler2.0"
    SFT_CKPT_DIR = BASE / "checkpoints_sft1_v51"
    BASE_CKPT_DIR = BASE / "checkpoints_v51"

sft_ckpts = sorted(SFT_CKPT_DIR.glob("sft_step_*.pt"))
base_ckpts = sorted(BASE_CKPT_DIR.glob("step_*.pt"))

print("=" * 50)
print("v51 SFT Stage 1 进度（纯 SFT 对话训练）")
print("=" * 50)

print(f"\n★ v51 Base checkpoint ({len(base_ckpts)} 个):")
if base_ckpts:
    target = BASE_CKPT_DIR / "step_015000.pt"
    if target.exists():
        sd = torch.load(target, map_location='cpu')
        print(f"  🎯 step_015000.pt | step={sd.get('step','?')} ema={sd.get('loss_ema','?'):.4f}")
    else:
        best = base_ckpts[-1]
        sd = torch.load(best, map_location='cpu')
        print(f"  最新: {best.name} | step={sd.get('step','?')} ema={sd.get('loss_ema','?'):.4f}")
else:
    print(f"  ⚠️ 无 — 请检查 {BASE_CKPT_DIR}")

print(f"\n★ SFT Stage 1 checkpoint ({len(sft_ckpts)} 个):")
if sft_ckpts:
    for c in sft_ckpts:
        sd = torch.load(c, map_location='cpu')
        mb = c.stat().st_size / 1e6
        st = sd.get('step', 0)
        le = sd.get('loss_ema', 0)
        pct = st / TC['max_steps'] * 100 if 'TC' in dir() else 0
        print(f"  {c.name:30s} step={st:>6} ({pct:.0f}%) ema={le:.4f} {mb:.0f}MB")
else:
    print(f"  尚无 — 运行 Cell 7 开始训练")

# Loss 日志摘要
log_file = SFT_CKPT_DIR / "loss_log.csv"
if log_file.exists():
    with open(log_file, 'r') as f:
        lines = f.readlines()
    print(f"\n📋 Loss 日志: {len(lines)-1} 行")
    if len(lines) > 11:
        print(f"  最近 10 步:")
        for line in lines[-10:]:
            print(f"    {line.rstrip()}")

print(f"\n{'='*50}")
print(f"目标: {TC['max_steps']} steps | ~1 epoch | 有效 batch={TC['micro_batch_size']}×{TC['grad_accum']}={TC['micro_batch_size']*TC['grad_accum']}")
print(f"数据: data_stage1/sft_pt/*.pt · ~1.74M 条")
print(f"Base: checkpoints_v51/step_015000.pt")
print(f"★ 训练完成后: chat.py 测试 → Stage 2 身份注入")